In [1]:

!nvidia-smi

Sat Nov 21 13:39:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas

In [4]:
import torch
import time
import pandas as pd
import gc

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Baseline FP16 model loaded successfully!")
print("Model device:", model_fp16.device)
print("GPU memory allocated:", round(
    torch.cuda.memory_allocated() / 1024**3, 2
), "GB")

GPU: Tesla T4


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Baseline FP16 model loaded successfully!
Model device: cuda:0
GPU memory allocated: 0.92 GB


In [5]:

# Baseline inference benchmark (FP16)

prompt = "Explain machine learning in simple terms with an example."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warm-up run
with torch.no_grad():
    _ = model_fp16.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Actual benchmark
start_time = time.perf_counter()

with torch.no_grad():
    output = model_fp16.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()
end_time = time.perf_counter()

generated_tokens = output.shape[1] - inputs["input_ids"].shape[1]
elapsed_time = end_time - start_time

baseline_speed = generated_tokens / elapsed_time
baseline_peak_memory = torch.cuda.max_memory_allocated() / (1024**3)

print("BASELINE FP16 RESULTS")
print("----------------------")
print("Generated tokens:", generated_tokens)
print("Time taken:", round(elapsed_time, 2), "seconds")
print("Tokens per second:", round(baseline_speed, 2))
print("Peak GPU memory:", round(baseline_peak_memory, 3), "GB")
print("\nGenerated response:")
print(tokenizer.decode(output[0], skip_special_tokens=True))


BASELINE FP16 RESULTS
----------------------
Generated tokens: 100
Time taken: 4.87 seconds
Tokens per second: 20.52
Peak GPU memory: 0.932 GB

Generated response:
Explain machine learning in simple terms with an example. Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time without being explicitly programmed. It involves using algorithms to identify patterns and make predictions or decisions based on the input data.

For example, consider a restaurant where you want to recommend dishes to your friends based on their past orders. You could use machine learning to analyze the customer data (such as age, gender, location, etc.) and other factors like cuisine preference, meal size, etc., to predict


In [6]:
# Release baseline model from GPU
del model_fp16

import gc
gc.collect()
torch.cuda.empty_cache()

print("Baseline model unloaded.")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

Baseline model unloaded.
GPU memory allocated: 0.008 GB


In [7]:
import torch
import time
import gc
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Configure 4-bit NF4 quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load optimized model
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

print("4-bit model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

4-bit model loaded successfully!
GPU: Tesla T4
GPU memory allocated: 0.434 GB


In [8]:

prompt = "Explain machine learning in simple terms with an example."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Warm-up
with torch.no_grad():
    _ = model_4bit.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Benchmark
start_time = time.perf_counter()

with torch.no_grad():
    output_4bit = model_4bit.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()
end_time = time.perf_counter()

generated_tokens_4bit = (
    output_4bit.shape[1] - inputs["input_ids"].shape[1]
)
elapsed_4bit = end_time - start_time

optimized_speed = generated_tokens_4bit / elapsed_4bit
optimized_peak_memory = (
    torch.cuda.max_memory_allocated() / (1024**3)
)

print("OPTIMIZED 4-BIT RESULTS")
print("-----------------------")
print("Generated tokens:", generated_tokens_4bit)
print("Time taken:", round(elapsed_4bit, 2), "seconds")
print("Tokens per second:", round(optimized_speed, 2))
print("Peak GPU memory:", round(optimized_peak_memory, 3), "GB")

print("\nGenerated response:")
print(tokenizer.decode(output_4bit[0], skip_special_tokens=True))


OPTIMIZED 4-BIT RESULTS
-----------------------
Generated tokens: 100
Time taken: 11.19 seconds
Tokens per second: 8.93
Peak GPU memory: 0.438 GB

Generated response:
Explain machine learning in simple terms with an example. Machine learning is a type of artificial intelligence that allows computers to learn from data and make predictions or decisions without being explicitly programmed. In other words, it's like teaching a computer how to solve a problem on its own based on the information it receives.

For example, imagine you have a set of images of cats and dogs. You want to teach your computer how to identify which image belongs to which animal. This is where machine learning comes in. The computer learns by analyzing each image and identifying patterns that


In [9]:

test_prompts = [
    "Explain artificial intelligence in simple terms.",
    "What is machine learning? Give one example.",
    "Explain the difference between AI and ML.",
    "What is Python used for?",
    "Explain a database with a simple example.",
    "What is deep learning?",
    "Explain overfitting in machine learning.",
    "What is an API and why is it useful?",
    "Explain neural networks in simple terms.",
    "What is natural language processing?"
]

# Generate 4-bit answers
results_4bit = []

for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model_4bit.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results_4bit.append(answer)
    print(f"\n{i}. PROMPT: {prompt}")
    print("4-BIT ANSWER:", answer)

print("\nAll 10 optimized answers collected!")



1. PROMPT: Explain artificial intelligence in simple terms.
4-BIT ANSWER:  Artificial Intelligence (AI) is a technology that allows machines to learn and perform tasks based on data, without being explicitly programmed. It's like having a smart assistant in your home who can understand what you say and do things for you, but it doesn't have the ability to make decisions or take action by itself. AI is used in many different ways, such as in self-driving cars, language translation, image recognition, and more. It's a rapidly growing field with potential applications in many industries.

2. PROMPT: What is machine learning? Give one example.
4-BIT ANSWER:  Machine learning is a type of artificial intelligence (AI) that enables computers to learn from data and improve their performance over time without being explicitly programmed. It involves the use of algorithms, statistical models, and other techniques to enable machines to recognize patterns in large amounts of data.

One example of

In [10]:

# Reload original FP16 model
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

results_fp16 = []

for i, prompt in enumerate(test_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model_fp16.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results_fp16.append(answer)

    print(f"\n{i}. PROMPT: {prompt}")
    print("FP16 ANSWER:", answer)

print("\nAll 10 FP16 answers collected!")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


1. PROMPT: Explain artificial intelligence in simple terms.
FP16 ANSWER:  Artificial Intelligence (AI) is a field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and language understanding.

In simpler terms, AI is the ability of computers to think like humans or mimic human behavior. It involves developing algorithms and machine learning techniques that allow computers to learn from data and make decisions based on that data. This allows computers to perform tasks that would be difficult or impossible for humans to do,

2. PROMPT: What is machine learning? Give one example.
FP16 ANSWER:  Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It can be used for a variety of tasks, including image recognition

In [11]:

comparison_df = pd.DataFrame({
    "Prompt": test_prompts,
    "FP16_Answer": results_fp16,
    "4bit_Answer": results_4bit
})

comparison_df.to_csv(
    "/content/task3_quality_comparison.csv",
    index=False
)

print("Comparison CSV saved successfully!")
print("Total prompts:", len(comparison_df))
display(comparison_df)


Comparison CSV saved successfully!
Total prompts: 10


,Prompt,FP16_Answer,4bit_Answer
0,Explain artificial intelligence in simple terms.,Artificial Intelligence (AI) is a field of co...,Artificial Intelligence (AI) is a technology ...
1,What is machine learning? Give one example.,Machine learning is a subset of artificial in...,Machine learning is a type of artificial inte...
2,Explain the difference between AI and ML.,"AI stands for Artificial Intelligence, which ...",Explain the difference between AI and ML.\n\n...
3,What is Python used for?,"Python is a high-level, interpreted programmi...",Python is a high-level programming language t...
4,Explain a database with a simple example.,A simple example of a database is a relationa...,A database is a collection of data organized ...
5,What is deep learning?,Deep learning is a subset of machine learning...,Deep learning is a type of machine learning t...
6,Explain overfitting in machine learning.,Overfitting occurs when a model is too comple...,Overfitting occurs when a model is trained on...
7,What is an API and why is it useful?,An API (Application Programming Interface) is...,An API (Application Programming Interface) is...
8,Explain neural networks in simple terms.,Neural networks are a type of machine learnin...,Neural networks are a type of machine learnin...
9,What is natural language processing?,Natural Language Processing (NLP) is a subfie...,Natural Language Processing (NLP) is a field ...


In [12]:
from google.colab import files
files.download("/content/task3_quality_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:

# Estimate KV cache memory for 20 concurrent users

config = model_fp16.config

layers = config.num_hidden_layers
kv_heads = config.num_key_value_heads
hidden_size = config.hidden_size
attention_heads = config.num_attention_heads

head_dim = hidden_size // attention_heads

# FP16 KV cache = 2 bytes per value
bytes_per_token = (
    2 * layers * kv_heads * head_dim * 2
)

context_length = 2048
concurrent_users = 20

kv_per_user = bytes_per_token * context_length
total_kv = kv_per_user * concurrent_users

kv_per_user_gb = kv_per_user / (1024**3)
total_kv_gb = total_kv / (1024**3)

# Use measured FP16 peak memory as a conservative
# model memory estimate for this experiment.
model_memory_gb = 0.932

estimated_total = model_memory_gb + total_kv_gb
gpu_capacity = 24

print("MODEL ARCHITECTURE")
print("Layers:", layers)
print("KV heads:", kv_heads)
print("Head dimension:", head_dim)

print("\n20-USER MEMORY ESTIMATE")
print("KV cache per user:", round(kv_per_user_gb, 4), "GB")
print("Total KV cache:", round(total_kv_gb, 3), "GB")
print("Model memory estimate:", model_memory_gb, "GB")
print("Combined estimate:", round(estimated_total, 3), "GB")
print("24GB GPU remaining:", round(gpu_capacity-estimated_total, 3), "GB")


MODEL ARCHITECTURE
Layers: 24
KV heads: 2
Head dimension: 64

20-USER MEMORY ESTIMATE
KV cache per user: 0.0234 GB
Total KV cache: 0.469 GB
Model memory estimate: 0.932 GB
Combined estimate: 1.401 GB
24GB GPU remaining: 22.599 GB


In [14]:

import pandas as pd

benchmark_results = pd.DataFrame([
    {
        "Configuration": "FP16 Baseline",
        "Quantization": "FP16",
        "Generated_Tokens": 100,
        "Time_Seconds": 4.87,
        "Tokens_Per_Second": 20.52,
        "Model_Load_Memory_GB": 0.920,
        "Peak_GPU_Memory_GB": 0.932
    },
    {
        "Configuration": "Optimized 4-bit",
        "Quantization": "NF4 4-bit",
        "Generated_Tokens": 100,
        "Time_Seconds": 11.19,
        "Tokens_Per_Second": 8.93,
        "Model_Load_Memory_GB": 0.434,
        "Peak_GPU_Memory_GB": 0.438
    }
])

benchmark_results.to_csv(
    "/content/task3_benchmark_results.csv",
    index=False
)

benchmark_results.to_csv(
    "/content/task3_benchmark_results.csv",
    index=False
)

print("Task 3 benchmark results saved!")
display(benchmark_results)


Task 3 benchmark results saved!


,Configuration,Quantization,Generated_Tokens,Time_Seconds,Tokens_Per_Second,Model_Load_Memory_GB,Peak_GPU_Memory_GB
0,FP16 Baseline,FP16,100,4.87,20.52,0.920,0.932
1,Optimized 4-bit,NF4 4-bit,100,11.19,8.93,0.434,0.438


In [15]:
from google.colab import files
files.download("/content/task3_benchmark_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>